# Cell 1: Import Dependencies

In [1]:
import sys
!{sys.executable} -m pip install -q pyarrow

In [2]:
import re
import os
import json
import time
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq


# Cell 2: Configuration

In [3]:
# ===== Input / Output =====
# Use a Path object to avoid relative-path confusion.
from pathlib import Path
INPUT_FILE = Path("../data/fineweb/004_00018.parquet")

# A short identifier for outputs (e.g., 004_00018)
SHARD_ID = INPUT_FILE.stem

# Cleaning versioning (v1 baseline; v2 adds spam/domain/very-long heuristics)
FILTER_VERSION = "v1"  # "v1" or "v2"

OUTPUT_DIR = Path("cleaning_outputs") / SHARD_ID / FILTER_VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_FILE = OUTPUT_DIR / f"{SHARD_ID}.clean.parquet"
REMOVED_FILE = OUTPUT_DIR / f"{SHARD_ID}.removed.parquet"
METRICS_FILE = OUTPUT_DIR / f"{SHARD_ID}.metrics_before_after.csv"
RULE_COUNTS_FILE = OUTPUT_DIR / f"{SHARD_ID}.rule_counts.csv"
IO_TIMING_FILE = OUTPUT_DIR / f"{SHARD_ID}.io_timing.csv"
RAW_SAMPLE_FILE = OUTPUT_DIR / f"{SHARD_ID}.raw_review_samples.csv"
CLEAN_SAMPLE_FILE = OUTPUT_DIR / f"{SHARD_ID}.clean_review_samples.csv"

# ===== Thresholds (tune as needed) =====
MIN_TOKEN_COUNT = 80
MIN_LANGUAGE_SCORE = 0.90
TARGET_LANGUAGE = "en"

# Optional v2 thresholds
MAX_TOKEN_COUNT = None          # e.g., 20000 to drop extreme long docs; keep None to disable
SPAM_KEYWORDS = [
    "casino", "wager", "betting", "bonus", "sportsbook", "slots",
    "crypto giveaway", "airdrop", "free bitcoin"
]
SPAM_KEYWORD_MATCH_MIN = 2      # how many keyword hits to consider spam-like

# Suspicious URL patterns (aggregation / weak-content pages)
SUSPICIOUS_URL_PATTERNS = [
    r"/tag/",
    r"/tags/",
    r"/category/",
    r"/author/",
    r"/page/\d+",
    r"\?replytocom=",
    r"\?amp",
    r"/wp-content/",
]

# Low-information / template-like patterns
LOW_INFO_PATTERNS = [
    r"\bprivacy policy\b",
    r"\bterms of service\b",
    r"\bcookie policy\b",
    r"\ball rights reserved\b",
    r"\bskip to content\b",
    r"\bclick here\b",
    r"\bsubscribe\b",
    r"\bsign up\b",
    r"\blog in\b",
    r"\bmenu\b",
    r"\bnavigation\b",
    r"\bhome\b",
    r"\bcontact us\b",
]

# Domain blacklist (keep empty by default; you can add confirmed low-quality domains)
BAD_DOMAINS = set()

# Review sampling
REVIEW_PER_BUCKET = 15  # total review size ~ buckets * REVIEW_PER_BUCKET (capped)
RANDOM_SEED = 42

# ===== Basic path sanity check =====
print("Current working directory:", os.getcwd())
print("Using input file:", INPUT_FILE)
print("Resolved path:", INPUT_FILE.resolve())
print("Exists:", INPUT_FILE.exists())


Current working directory: /home/jovyan/week3
Using input file: ../data/fineweb/004_00018.parquet
Resolved path: /home/jovyan/data/fineweb/004_00018.parquet
Exists: True


# Cell 3: Load Data

In [4]:
# Load the shard (timed)
io_timing = {"shard_id": SHARD_ID, "filter_version": FILTER_VERSION}

t0 = time.perf_counter()
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"INPUT_FILE not found: {INPUT_FILE.resolve()}")

# Read metadata quickly
t_meta0 = time.perf_counter()
pf = pq.ParquetFile(str(INPUT_FILE))
io_timing["read_meta_s"] = time.perf_counter() - t_meta0
io_timing["row_groups"] = pf.num_row_groups
io_timing["rows"] = pf.metadata.num_rows
io_timing["columns"] = pf.metadata.num_columns

# Load as dataset/table
t_load0 = time.perf_counter()
dataset = ds.dataset(str(INPUT_FILE), format="parquet")
table = dataset.to_table()
io_timing["load_table_s"] = time.perf_counter() - t_load0

t_pd0 = time.perf_counter()
df = table.to_pandas()
io_timing["to_pandas_s"] = time.perf_counter() - t_pd0

io_timing["total_load_s"] = time.perf_counter() - t0

print("rows:", len(df))
print("columns:", list(df.columns))
df.head(3)


rows: 172447
columns: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count']


,text,id,dump,url,date,file_path,language,language_score,token_count
0,Welcome to Harrisonburg TownSquare.\nTownSquar...,<urn:uuid:189514cf-784a-4e9f-9e8e-f350c5a41adf>,CC-MAIN-2025-26,https://www.townsquareapp.co/,2025-06-16T16:22:40Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.899284,203
1,"The Art of Effective Board Reporting\nSep 5, 2...",<urn:uuid:9d2f701d-48f5-418f-a8e2-d5b2d51aad36>,CC-MAIN-2025-26,https://www.traact.com/blog/the-art-of-effecti...,2025-06-16T16:45:14Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.942698,2717
2,Belinda's journey and transformation is a real...,<urn:uuid:ccb16b4e-f301-480c-9231-33f6e823eabe>,CC-MAIN-2025-26,https://www.tracydixonmindandbodyfit.com/post/...,2025-06-16T15:44:36Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.987144,555


# Cell 4: Helper Functions

In [5]:
def normalize_text(x: object) -> str:
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\x00", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def get_domain(url: object) -> str:
    try:
        return urlparse(str(url)).netloc.lower()
    except Exception:
        return ""

def is_empty_or_almost_empty(text: str) -> bool:
    if not text:
        return True
    core = re.sub(r"[^A-Za-z0-9]+", "", text)
    return len(core) < 20

def has_suspicious_url(url: object) -> bool:
    u = str(url).lower()
    return any(re.search(pat, u) for pat in SUSPICIOUS_URL_PATTERNS)

def repeated_char_ratio(text: str) -> float:
    if not text:
        return 1.0
    chars = [c for c in text if not c.isspace()]
    if not chars:
        return 1.0
    from collections import Counter
    cnt = Counter(chars)
    return max(cnt.values()) / len(chars)

def repeated_ngram_ratio(text: str, n: int = 3) -> float:
    words = text.lower().split()
    if len(words) < n * 2:
        return 0.0
    ngrams = [" ".join(words[i:i+n]) for i in range(len(words) - n + 1)]
    if not ngrams:
        return 0.0
    from collections import Counter
    cnt = Counter(ngrams)
    return max(cnt.values()) / len(ngrams)

def is_low_information(text: str) -> bool:
    text_l = text.lower()
    keyword_hits = sum(bool(re.search(pat, text_l)) for pat in LOW_INFO_PATTERNS)
    char_rep = repeated_char_ratio(text)
    ngram_rep = repeated_ngram_ratio(text, n=3)
    words = re.findall(r"[A-Za-z]+", text_l)
    uniq_ratio = len(set(words)) / len(words) if words else 0.0

    return (
        keyword_hits >= 3
        or char_rep > 0.35
        or ngram_rep > 0.08
        or (len(words) >= 30 and uniq_ratio < 0.22)
    )

def exact_text_key(text: str) -> str:
    return normalize_text(text).lower()

def spam_keyword_hits(text: str) -> int:
    if not SPAM_KEYWORDS:
        return 0
    t = text.lower()
    return sum(1 for kw in SPAM_KEYWORDS if kw in t)

def is_spam_like(text: str) -> bool:
    return spam_keyword_hits(text) >= SPAM_KEYWORD_MATCH_MIN

def bucket_token_count(tc: float) -> str:
    if pd.isna(tc):
        return "unknown"
    tc = float(tc)
    if tc < 100:
        return "short_<100"
    if tc < 500:
        return "mid_100_500"
    if tc < 2000:
        return "long_500_2000"
    if tc < 5000:
        return "very_long_2000_5000"
    return "extreme_>=5000"

def bucket_language_score(ls: float) -> str:
    if pd.isna(ls):
        return "unknown"
    ls = float(ls)
    if ls < 0.85:
        return "low_<0.85"
    if ls < 0.90:
        return "mid_0.85_0.90"
    return "high_>=0.90"

def stratified_sample(df_in: pd.DataFrame, n_per_bucket: int, seed: int, label: str) -> pd.DataFrame:
    # Create buckets
    tmp = df_in.copy()
    tmp["bucket_token"] = tmp["token_count"].apply(bucket_token_count)
    tmp["bucket_lang"] = tmp["language_score"].apply(bucket_language_score)
    tmp["bucket_urlpattern"] = tmp["url"].apply(lambda u: "weak_urlpattern" if has_suspicious_url(u) else "normal_url")
    tmp["bucket"] = tmp["bucket_token"] + "|" + tmp["bucket_lang"] + "|" + tmp["bucket_urlpattern"]

    rng = np.random.default_rng(seed)
    samples = []
    for b, g in tmp.groupby("bucket", sort=False):
        k = min(n_per_bucket, len(g))
        if k <= 0:
            continue
        idx = rng.choice(g.index.to_numpy(), size=k, replace=False)
        samples.append(tmp.loc[idx])

    if not samples:
        return tmp.head(0)

    out = pd.concat(samples, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    out["sample_set"] = label
    return out


# Cell 5: Feature Preparation

In [6]:
t_feat0 = time.perf_counter()

df = df.copy()

df["text_norm"] = df["text"].apply(normalize_text)
df["domain"] = df["url"].apply(get_domain)
df["weak_urlpattern"] = df["url"].apply(has_suspicious_url)

# Standardize types
for col in ["id", "dump", "url", "date", "file_path", "language"]:
    df[col] = df[col].astype("string")

df["language_score"] = pd.to_numeric(df["language_score"], errors="coerce")
df["token_count"] = pd.to_numeric(df["token_count"], errors="coerce")

io_timing["feature_engineering_s"] = time.perf_counter() - t_feat0

df[["language", "language_score", "token_count", "domain", "weak_urlpattern"]].head(3)


,language,language_score,token_count,domain,weak_urlpattern
0,en,0.899284,203,www.townsquareapp.co,False
1,en,0.942698,2717,www.traact.com,False
2,en,0.987144,555,www.tracydixonmindandbodyfit.com,False


# Cell 6: Mark Removal Reasons

In [7]:
# Build rule masks (v1 baseline + optional v2 additions)

# v1 rules
rule_empty_text = df["text_norm"].apply(is_empty_or_almost_empty)
rule_short_text = df["token_count"].fillna(0) < MIN_TOKEN_COUNT
rule_non_english = df["language"].fillna("").str.lower() != TARGET_LANGUAGE
rule_low_lang_score = df["language_score"].fillna(0) < MIN_LANGUAGE_SCORE
rule_bad_domain = df["domain"].isin(BAD_DOMAINS)
rule_suspicious_url = df["weak_urlpattern"]
rule_low_info = df["text_norm"].apply(is_low_information)

# v1 dedup (exact)
rule_dup_id = df["id"].duplicated(keep="first")
rule_dup_url = df["url"].duplicated(keep="first")
text_keys = df["text_norm"].apply(exact_text_key)
rule_dup_text = text_keys.duplicated(keep="first")

rule_map = {
    "empty_or_almost_empty_text": rule_empty_text,
    "token_count_too_low": rule_short_text,
    "non_english_language": rule_non_english,
    "language_score_too_low": rule_low_lang_score,
    "bad_domain": rule_bad_domain,
    "suspicious_url_pattern": rule_suspicious_url,
    "low_information_or_template_page": rule_low_info,
    "duplicate_id": rule_dup_id,
    "duplicate_url": rule_dup_url,
    "duplicate_text_exact": rule_dup_text,
}

# v2 rules (optional)
if FILTER_VERSION.lower() == "v2":
    if MAX_TOKEN_COUNT is not None:
        rule_map["token_count_too_high"] = df["token_count"].fillna(0) > float(MAX_TOKEN_COUNT)
    # spam keyword heuristic (long-but-noisy pages)
    rule_map["spam_keyword_content"] = df["text_norm"].apply(is_spam_like)

# Create per-rule boolean columns for audit and compute a combined reason string
reason_cols = []
for rule_name, mask in rule_map.items():
    col_name = f"rm_{rule_name}"
    df[col_name] = mask
    reason_cols.append(col_name)

def collect_reasons(row):
    reasons = []
    for rule_name in rule_map.keys():
        if row[f"rm_{rule_name}"]:
            reasons.append(rule_name)
    return "|".join(reasons)

df["remove_reasons"] = df.apply(collect_reasons, axis=1)
df["is_removed"] = df["remove_reasons"] != ""

print(df["is_removed"].value_counts())


is_removed
False    142316
True      30131
Name: count, dtype: int64


# Cell 7: Build Cleaned and Removed Datasets

In [8]:
clean_df = df.loc[~df["is_removed"]].copy()
removed_df = df.loc[df["is_removed"]].copy()

print("raw rows   :", len(df))
print("clean rows :", len(clean_df))
print("removed    :", len(removed_df))
print("remove rate:", len(removed_df) / len(df))

raw rows   : 172447
clean rows : 142316
removed    : 30131
remove rate: 0.17472614774394452


# Cell 8: Evaluation Function

In [9]:
def dataset_metrics(x: pd.DataFrame, name: str) -> dict:
    out = {}
    out["dataset"] = name
    out["doc_count"] = len(x)

    # Length
    out["avg_token_count"] = x["token_count"].mean()
    out["median_token_count"] = x["token_count"].median()
    out["p25_token_count"] = x["token_count"].quantile(0.25)
    out["p75_token_count"] = x["token_count"].quantile(0.75)
    out["p90_token_count"] = x["token_count"].quantile(0.90)
    out["p95_token_count"] = x["token_count"].quantile(0.95)
    out["p99_token_count"] = x["token_count"].quantile(0.99)

    out["short_lt_50_ratio"] = (x["token_count"] < 50).mean()
    out["short_lt_100_ratio"] = (x["token_count"] < 100).mean()
    out["short_lt_200_ratio"] = (x["token_count"] < 200).mean()
    out["long_gt_2000_ratio"] = (x["token_count"] > 2000).mean()
    out["long_gt_5000_ratio"] = (x["token_count"] > 5000).mean()

    # Language
    out["english_ratio"] = (x["language"].fillna("").str.lower() == "en").mean()
    out["avg_language_score"] = x["language_score"].mean()
    out["median_language_score"] = x["language_score"].median()
    out["lang_score_lt_08_ratio"] = (x["language_score"] < 0.8).mean()
    out["lang_score_lt_09_ratio"] = (x["language_score"] < 0.9).mean()
    out["lang_score_lt_085_ratio"] = (x["language_score"] < 0.85).mean()

    # Duplicates (exact)
    out["duplicate_id_ratio"] = x["id"].duplicated().mean()
    out["duplicate_url_ratio"] = x["url"].duplicated().mean()
    out["duplicate_text_ratio"] = x["text_norm"].apply(exact_text_key).duplicated().mean()

    # Source / URL patterns
    out["unique_domain_count"] = x["domain"].nunique()
    out["weak_urlpattern_ratio"] = x["weak_urlpattern"].mean() if "weak_urlpattern" in x.columns else np.nan

    # v2 spam heuristic indicator (computed on the fly; can be expensive on huge data)
    out["spam_like_ratio"] = x["text_norm"].apply(is_spam_like).mean() if FILTER_VERSION.lower()=="v2" else np.nan

    return out


# Cell 9: Compute Before/After Quality Metrics

In [10]:
raw_metrics = dataset_metrics(df, "raw")
clean_metrics = dataset_metrics(clean_df, "clean")

metrics_df = pd.DataFrame([raw_metrics, clean_metrics])

raw_count = len(df)
metrics_df["retention_ratio"] = metrics_df["doc_count"] / raw_count
metrics_df["deletion_ratio"] = 1 - metrics_df["retention_ratio"]

metrics_df["shard_id"] = SHARD_ID
metrics_df["filter_version"] = FILTER_VERSION

metrics_df


,dataset,doc_count,avg_token_count,median_token_count,p25_token_count,p75_token_count,p90_token_count,p95_token_count,p99_token_count,short_lt_50_ratio,...,duplicate_id_ratio,duplicate_url_ratio,duplicate_text_ratio,unique_domain_count,weak_urlpattern_ratio,spam_like_ratio,retention_ratio,deletion_ratio,shard_id,filter_version
0,raw,172447,779.813833,487.0,241.0,916.0,1611.0,2307.0,4865.08,0.000429,...,0.0,0.000087,0.0,139653,0.021317,NaN,1.000000,0.000000,004_00018,v1
1,clean,142316,774.271923,530.0,276.0,957.0,1620.0,2238.0,4135.00,0.000000,...,0.0,0.000000,0.0,115465,0.000000,NaN,0.825274,0.174726,004_00018,v1


# Cell 10: Count Removals by Rule

In [11]:
rule_counts = []
raw_n = len(df)
removed_n = int(df["is_removed"].sum())

for rule_name in rule_map.keys():
    cnt = int(df[f"rm_{rule_name}"].sum())
    rule_counts.append({
        "shard_id": SHARD_ID,
        "filter_version": FILTER_VERSION,
        "rule": rule_name,
        "hit_count": cnt,  # rule hits (not mutually exclusive)
        "hit_ratio_over_raw": cnt / raw_n,
    })

rule_counts_df = pd.DataFrame(rule_counts).sort_values("hit_count", ascending=False)

# Add a summary row for total removed (unique records removed)
summary_row = pd.DataFrame([{
    "shard_id": SHARD_ID,
    "filter_version": FILTER_VERSION,
    "rule": "__TOTAL_REMOVED__",
    "hit_count": removed_n,
    "hit_ratio_over_raw": removed_n / raw_n,
}])
rule_counts_df = pd.concat([summary_row, rule_counts_df], ignore_index=True)

rule_counts_df


,shard_id,filter_version,rule,hit_count,hit_ratio_over_raw
0,004_00018,v1,__TOTAL_REMOVED__,30131,0.174726
1,004_00018,v1,language_score_too_low,24048,0.139452
2,004_00018,v1,suspicious_url_pattern,3676,0.021317
3,004_00018,v1,token_count_too_low,2516,0.014590
4,004_00018,v1,low_information_or_template_page,1809,0.010490
5,004_00018,v1,duplicate_url,15,0.000087
6,004_00018,v1,empty_or_almost_empty_text,0,0.000000
7,004_00018,v1,bad_domain,0,0.000000
8,004_00018,v1,non_english_language,0,0.000000
9,004_00018,v1,duplicate_id,0,0.000000


# Cell 11: Source Quality Comparison (Top Domains)

In [12]:
raw_domain_top = df["domain"].value_counts().head(20).rename("raw_top_domains")
clean_domain_top = clean_df["domain"].value_counts().head(20).rename("clean_top_domains")

print("=== Raw top domains ===")
display(raw_domain_top)

print("\n=== Clean top domains ===")
display(clean_domain_top)

=== Raw top domains ===


domain
pubmed.ncbi.nlm.nih.gov        59
plandisney.disney.go.com       53
www.telegraph.co.uk            46
www.ibtimes.co.uk              43
www.cbsnews.com                42
www.latimes.com                38
www.sciencedaily.com           38
www.prweb.com                  38
timesofindia.indiatimes.com    37
publications.parliament.uk     33
www.techradar.com              32
www.foxnews.com                32
www.gofundme.com               32
community.oracle.com           29
www.csmonitor.com              29
www.aljazeera.com              29
dev.to                         28
www.jalopnik.com               27
www.digitaltrends.com          27
forum.trains.com               26
Name: raw_top_domains, dtype: int64


=== Clean top domains ===


domain
plandisney.disney.go.com       50
www.telegraph.co.uk            46
pubmed.ncbi.nlm.nih.gov        43
www.ibtimes.co.uk              43
www.cbsnews.com                41
www.latimes.com                38
www.sciencedaily.com           38
timesofindia.indiatimes.com    37
www.prweb.com                  36
www.foxnews.com                32
www.gofundme.com               32
publications.parliament.uk     31
www.csmonitor.com              29
www.aljazeera.com              28
www.techradar.com              27
www.jalopnik.com               27
www.digitaltrends.com          27
forum.trains.com               26
www.themoscowtimes.com         25
www.business-standard.com      25
Name: clean_top_domains, dtype: int64

# Cell 12: Export Manual Review Samples (50 Each)

In [13]:
# Export stratified review samples (more informative than pure random sampling)
raw_review = stratified_sample(
    df[["text", "url", "domain", "language", "language_score", "token_count", "weak_urlpattern", "text_norm"]],
    n_per_bucket=REVIEW_PER_BUCKET,
    seed=RANDOM_SEED,
    label="raw",
)

clean_review = stratified_sample(
    clean_df[["text", "url", "domain", "language", "language_score", "token_count", "weak_urlpattern", "text_norm"]],
    n_per_bucket=REVIEW_PER_BUCKET,
    seed=RANDOM_SEED,
    label="clean",
)

# Keep a compact set of columns for manual inspection
keep_cols = ["sample_set", "bucket", "bucket_token", "bucket_lang", "bucket_urlpattern",
             "url", "domain", "language", "language_score", "token_count", "text"]

raw_sample = raw_review[keep_cols].copy()
clean_sample = clean_review[keep_cols].copy()

# Add manual annotation columns
for s in [raw_sample, clean_sample]:
    s["manual_label"] = ""
    s["manual_notes"] = ""

raw_sample.to_csv(RAW_SAMPLE_FILE, index=False)
clean_sample.to_csv(CLEAN_SAMPLE_FILE, index=False)

display(raw_sample.head(10))
display(clean_sample.head(10))


,sample_set,bucket,bucket_token,bucket_lang,bucket_urlpattern,url,domain,language,language_score,token_count,text,manual_label,manual_notes
0,raw,long_500_2000|mid_0.85_0.90|normal_url,long_500_2000,mid_0.85_0.90,normal_url,https://tinygrab.com/how-do-you-set-up-an-acco...,tinygrab.com,en,0.890701,1631,How to Set Up an Amazon Account: A Comprehensi...,,
1,raw,mid_100_500|high_>=0.90|weak_urlpattern,mid_100_500,high_>=0.90,weak_urlpattern,https://mediinkllc.com/author/aspwv-com/,mediinkllc.com,en,0.946508,108,How to Get a Safer Massage During Coronavirus\...,,
2,raw,very_long_2000_5000|mid_0.85_0.90|normal_url,very_long_2000_5000,mid_0.85_0.90,normal_url,https://evrokontakt.ru/hotel/2314/,evrokontakt.ru,en,0.864586,2232,Rosewood has long been synonymous with barefoo...,,
3,raw,long_500_2000|high_>=0.90|normal_url,long_500_2000,high_>=0.90,normal_url,https://motorcitymuckraker.com/2013/02/06/mayo...,motorcitymuckraker.com,en,0.967436,850,They said the first-term legislator couldn’t r...,,
4,raw,long_500_2000|high_>=0.90|normal_url,long_500_2000,high_>=0.90,normal_url,https://www.tipsnsolution.in/online-gas-booking/,www.tipsnsolution.in,en,0.927958,1298,"We have three choices for LPG gases, Hp, Bhara...",,
5,raw,short_<100|mid_0.85_0.90|normal_url,short_<100,mid_0.85_0.90,normal_url,https://newyou.com.au/treatments/mela-peel/,newyou.com.au,en,0.871024,89,Embrace your natural beauty and discover attai...,,
6,raw,long_500_2000|low_<0.85|normal_url,long_500_2000,low_<0.85,normal_url,https://www.atomicmatters.eu/en/english-comput...,www.atomicmatters.eu,en,0.845903,1073,All ATOMIC MATTERS calculations can simulate m...,,
7,raw,extreme_>=5000|mid_0.85_0.90|normal_url,extreme_>=5000,mid_0.85_0.90,normal_url,http://bethge-family.de/pdf.php?q=download-int...,bethge-family.de,en,0.899772,36937,Download International Review Of Neurobiology ...,,
8,raw,extreme_>=5000|high_>=0.90|normal_url,extreme_>=5000,high_>=0.90,normal_url,https://abandonedin360.com/abandoned-residenti...,abandonedin360.com,en,0.926437,5319,Chateaux La Pointe Koenig: Unraveling the Haun...,,
9,raw,very_long_2000_5000|mid_0.85_0.90|normal_url,very_long_2000_5000,mid_0.85_0.90,normal_url,https://uberweedshop.co/fr/brampton-weed-deliv...,uberweedshop.co,en,0.880554,2311,Same Day Weed Delivery in Brampton\nSame day h...,,


,sample_set,bucket,bucket_token,bucket_lang,bucket_urlpattern,url,domain,language,language_score,token_count,text,manual_label,manual_notes
0,clean,very_long_2000_5000|high_>=0.90|normal_url,very_long_2000_5000,high_>=0.90,normal_url,https://www.guitarworld.com/features/robert-fr...,www.guitarworld.com,en,0.933361,4684,Robert Fripp is widely hailed as the king of p...,,
1,clean,extreme_>=5000|high_>=0.90|normal_url,extreme_>=5000,high_>=0.90,normal_url,https://www.raystedman.org/zh/old-testament/is...,www.raystedman.org,en,0.976623,5051,Man's Insignificance -- God's Majesty | |\nThe...,,
2,clean,very_long_2000_5000|high_>=0.90|normal_url,very_long_2000_5000,high_>=0.90,normal_url,https://marxist.com/britain-labour-socialist-p...,marxist.com,en,0.965018,3503,Labour victory - Now fight for Socialist Polic...,,
3,clean,very_long_2000_5000|high_>=0.90|normal_url,very_long_2000_5000,high_>=0.90,normal_url,https://travelacharya.in/gopeshwar-temple-utta...,travelacharya.in,en,0.962266,2584,"Gopeshwar, the town in the name of Lord Shiva ...",,
4,clean,mid_100_500|high_>=0.90|normal_url,mid_100_500,high_>=0.90,normal_url,https://en.alexandragorsche.at/food/crispy-ric...,en.alexandragorsche.at,en,0.923508,370,Craving a crispy surprise from the oven? This ...,,
5,clean,extreme_>=5000|high_>=0.90|normal_url,extreme_>=5000,high_>=0.90,normal_url,https://www.ewtn.com/catholicism/library/latin...,www.ewtn.com,en,0.945024,6352,Latin and Vernacular: Language in the Roman Li...,,
6,clean,long_500_2000|high_>=0.90|normal_url,long_500_2000,high_>=0.90,normal_url,https://thebusinessgroup.co.uk/brighton-honour...,thebusinessgroup.co.uk,en,0.953527,893,Brighton honoured in Curry Capital of Britain ...,,
7,clean,very_long_2000_5000|high_>=0.90|normal_url,very_long_2000_5000,high_>=0.90,normal_url,https://www.imphaltimes.com/other-news/mcw-exc...,www.imphaltimes.com,en,0.931610,2397,Rising Popularity of Betting Alternatives\nIn ...,,
8,clean,extreme_>=5000|high_>=0.90|normal_url,extreme_>=5000,high_>=0.90,normal_url,https://gitech.ac.id/slottica-registrace-best-...,gitech.ac.id,en,0.939056,6313,Mostbet Games Inc oversees all of Mostbet Casi...,,
9,clean,extreme_>=5000|high_>=0.90|normal_url,extreme_>=5000,high_>=0.90,normal_url,https://www.alicespringsnews.com.au/archive/13...,www.alicespringsnews.com.au,en,0.968689,10069,"ALICE SPRINGS NEWS,\nAugust 31, 2006. This pag...",,


# Cell 13: Save Outputs

In [14]:
# Drop helper columns before saving main parquet outputs
helper_cols = ["text_norm", "domain", "weak_urlpattern", "remove_reasons", "is_removed"] + reason_cols
save_clean_df = clean_df.drop(columns=[c for c in helper_cols if c in clean_df.columns], errors="ignore")
save_removed_df = removed_df.copy()

# Save parquet outputs
save_clean_df.to_parquet(CLEAN_FILE, index=False)
save_removed_df.to_parquet(REMOVED_FILE, index=False)

# Save metrics / rule counts / IO timing
metrics_df.to_csv(METRICS_FILE, index=False)
rule_counts_df.to_csv(RULE_COUNTS_FILE, index=False)

# Save IO timing as one-row CSV
pd.DataFrame([io_timing]).to_csv(IO_TIMING_FILE, index=False)

print("Saved outputs to:", OUTPUT_DIR)
print(" clean data   ->", CLEAN_FILE)
print(" removed data ->", REMOVED_FILE)
print(" metrics      ->", METRICS_FILE)
print(" rule counts  ->", RULE_COUNTS_FILE)
print(" io timing    ->", IO_TIMING_FILE)
print(" raw samples  ->", RAW_SAMPLE_FILE)
print(" clean samples->", CLEAN_SAMPLE_FILE)


Saved outputs to: cleaning_outputs/004_00018/v1
 clean data   -> cleaning_outputs/004_00018/v1/004_00018.clean.parquet
 removed data -> cleaning_outputs/004_00018/v1/004_00018.removed.parquet
 metrics      -> cleaning_outputs/004_00018/v1/004_00018.metrics_before_after.csv
 rule counts  -> cleaning_outputs/004_00018/v1/004_00018.rule_counts.csv
 io timing    -> cleaning_outputs/004_00018/v1/004_00018.io_timing.csv
 raw samples  -> cleaning_outputs/004_00018/v1/004_00018.raw_review_samples.csv
 clean samples-> cleaning_outputs/004_00018/v1/004_00018.clean_review_samples.csv


# Cell 14: Print Summary

In [15]:
print("=== Summary ===")
print("Shard:", SHARD_ID, "| Version:", FILTER_VERSION)
print(f"Raw docs:     {len(df):,}")
print(f"Clean docs:   {len(clean_df):,}")
print(f"Removed docs: {len(removed_df):,}")
print(f"Retention:    {len(clean_df)/len(df):.2%}")
print(f"Deletion:     {len(removed_df)/len(df):.2%}")

print("\n=== IO Timing (seconds) ===")
for k in ["read_meta_s", "load_table_s", "to_pandas_s", "feature_engineering_s", "total_load_s"]:
    if k in io_timing:
        print(f"{k}: {io_timing[k]:.4f}")

print("\n=== Before/After Metrics ===")
display(metrics_df)

print("\n=== Rule Hits (Top) ===")
display(rule_counts_df.head(15))


=== Summary ===
Shard: 004_00018 | Version: v1
Raw docs:     172,447
Clean docs:   142,316
Removed docs: 30,131
Retention:    82.53%
Deletion:     17.47%

=== IO Timing (seconds) ===
read_meta_s: 0.0043
load_table_s: 0.6949
to_pandas_s: 0.0069
feature_engineering_s: 37.8622
total_load_s: 0.7063

=== Before/After Metrics ===


,dataset,doc_count,avg_token_count,median_token_count,p25_token_count,p75_token_count,p90_token_count,p95_token_count,p99_token_count,short_lt_50_ratio,...,duplicate_id_ratio,duplicate_url_ratio,duplicate_text_ratio,unique_domain_count,weak_urlpattern_ratio,spam_like_ratio,retention_ratio,deletion_ratio,shard_id,filter_version
0,raw,172447,779.813833,487.0,241.0,916.0,1611.0,2307.0,4865.08,0.000429,...,0.0,0.000087,0.0,139653,0.021317,NaN,1.000000,0.000000,004_00018,v1
1,clean,142316,774.271923,530.0,276.0,957.0,1620.0,2238.0,4135.00,0.000000,...,0.0,0.000000,0.0,115465,0.000000,NaN,0.825274,0.174726,004_00018,v1



=== Rule Hits (Top) ===


,shard_id,filter_version,rule,hit_count,hit_ratio_over_raw
0,004_00018,v1,__TOTAL_REMOVED__,30131,0.174726
1,004_00018,v1,language_score_too_low,24048,0.139452
2,004_00018,v1,suspicious_url_pattern,3676,0.021317
3,004_00018,v1,token_count_too_low,2516,0.014590
4,004_00018,v1,low_information_or_template_page,1809,0.010490
5,004_00018,v1,duplicate_url,15,0.000087
6,004_00018,v1,empty_or_almost_empty_text,0,0.000000
7,004_00018,v1,bad_domain,0,0.000000
8,004_00018,v1,non_english_language,0,0.000000
9,004_00018,v1,duplicate_id,0,0.000000
